# Four-Format Data Recovery and Validation
## Group 5 — Final Technical Report

**Dataset:** Gapminder Historical Teaching Snapshot

This notebook documents the recovery and validation of historical country observations from CSV, Excel, JSON, and Parquet sources.

It presents recovery counts, cross-format agreement, canonical exports, NumPy calculations, and interpretations of the results.

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

from src.pipeline import run_pipeline

pd.set_option("display.max_columns", None)

## 1. Data Recovery and Validation

The project reads four source formats, verifies their integrity using SHA-256 checksums, maps source fields into a common schema, validates records, quarantines invalid rows, and removes exact duplicates.

The recovered records are compared across formats before the canonical dataset is published.

In [2]:
with open("config.json", encoding="utf-8") as file:
    config = json.load(file)

run_pipeline(config)

Data directory: data
Output directory: output
Reporting year: 2007

Reading: country_records.csv
SHA-256: PASS
Rows loaded: 1712
Column mapping: PASS

Quarantined: 6
Duplicates removed: 2
Clean records: 1704
Validation: PASS

Reading: country_records.xlsx
SHA-256: PASS
Rows loaded: 1712
Column mapping: PASS

Quarantined: 6
Duplicates removed: 2
Clean records: 1704
Validation: PASS

Reading: country_records.json
SHA-256: PASS
Rows loaded: 1712
Column mapping: PASS

Quarantined: 6
Duplicates removed: 2
Clean records: 1704
Validation: PASS

Reading: country_records.parquet
SHA-256: PASS
Rows loaded: 1712
Column mapping: PASS

Quarantined: 6
Duplicates removed: 2
Clean records: 1704
Validation: PASS

All four formats validated successfully!
Datasets validated: 4
Total quarantined: 24
Total duplicates removed: 8
All validation counts: PASS

Starting four-format comparison...
CSV vs Excel: PASS
CSV vs JSON: PASS
CSV vs Parquet: PASS

Four-format agreement: PASS
Final canonical records: 1704


In [3]:
output = Path("output")

validation = pd.read_csv(output / "validation.csv")
quarantine = pd.read_csv(output / "quarantine.csv")
duplicates = pd.read_csv(output / "duplicate_log.csv")
canonical = pd.read_csv(output / "canonical.csv")

print("Canonical records:", len(canonical))
print("Quarantined records:", len(quarantine))
print("Exact duplicates removed:", len(duplicates))

display(validation)

Canonical records: 1704
Quarantined records: 24
Exact duplicates removed: 8


,check,expected,actual,tolerance,pass
0,country_records.csv: raw rows,1712,1712,0,True
1,country_records.csv: rejected rows,6,6,0,True
2,country_records.csv: duplicates,2,2,0,True
3,country_records.csv: canonical rows,1704,1704,0,True
4,country_records.csv: reconciliation,True,True,0,True
5,country_records.xlsx: raw rows,1712,1712,0,True
6,country_records.xlsx: rejected rows,6,6,0,True
7,country_records.xlsx: duplicates,2,2,0,True
8,country_records.xlsx: canonical rows,1704,1704,0,True
9,country_records.xlsx: reconciliation,True,True,0,True


## 2. Recovery Interpretation

Each source contains 1,712 raw records. The validation process identifies six invalid records and removes two exact duplicates per source, leaving 1,704 valid records per format.

The canonical dataset contains 1,704 records because the four source files represent the same observations rather than four independent datasets.

In [4]:
agreement = validation[
    validation["check"].astype(str).str.contains(
        "agreement|comparison|round.trip|roundtrip",
        case=False,
        regex=True
    )
]

display(agreement)

,check,expected,actual,tolerance,pass
22,CSV vs Excel agreement,True,True,0,True
23,CSV vs JSON agreement,True,True,0,True
24,CSV vs Parquet agreement,True,True,0,True
34,Pandas NumPy Asia total agreement,3811953827,3811953827,0,True
36,CSV export roundtrip,True,True,0,True
37,XLSX export roundtrip,True,True,0,True
38,JSON export roundtrip,True,True,0,True
39,PARQUET export roundtrip,True,True,0,True


In [5]:
asia = pd.read_csv(output / "asia_selected_year.csv")
history = pd.read_csv(output / "country_history.csv")
population_index = pd.read_csv(output / "population_index.csv")

asia_population = np.sum(asia["pop"].to_numpy(dtype=np.int64))

print("Asia selected-year population:", asia_population)
print("Philippines historical rows:", len(history))

display(population_index)

Asia selected-year population: 3811953827
Philippines historical rows: 12


,country,1982,1987,1992,1997,2002,2007
0,Philippines,100.0,112.273494,125.682418,140.324570,155.256447,170.375577
1,Indonesia,100.0,110.390432,120.524576,129.955720,137.639149,145.782331
2,Thailand,100.0,108.362522,116.056504,123.326192,128.630762,133.262203


## 3. Analytical Interpretation

The selected-year Asia report demonstrates filtering and aggregation using the recovered canonical records.

The country history report provides a longitudinal view of the Philippines, while the population index presents selected countries across historical years.

NumPy calculations are used to verify population totals and support reproducible numerical analysis.

These observations describe the historical Gapminder teaching dataset and should not be interpreted as current population estimates.

In [6]:
audit = pd.read_csv(output / "audit_log.csv")

display(audit)
display(validation)

if "pass" in validation.columns:
    passed = validation["pass"].astype(str).str.lower().eq("true")
    print("Validation checks passed:", int(passed.sum()))
    print("Total validation checks:", len(validation))
    print("All checks passed:", bool(passed.all()))

,step,source_file,operation,rule,rows_before,rows_after
0,reading,country_records.csv,read_and_map,SHA-256 verified; canonical columns mapped,1712,1712
1,validation,country_records.csv,quarantine_invalid,Manifest schema and domain rules,1712,1706
2,deduplication,country_records.csv,remove_exact_duplicates,Exact duplicate records removed,1706,1704
3,reading,country_records.xlsx,read_and_map,SHA-256 verified; canonical columns mapped,1712,1712
4,validation,country_records.xlsx,quarantine_invalid,Manifest schema and domain rules,1712,1706
5,deduplication,country_records.xlsx,remove_exact_duplicates,Exact duplicate records removed,1706,1704
6,reading,country_records.json,read_and_map,SHA-256 verified; canonical columns mapped,1712,1712
7,validation,country_records.json,quarantine_invalid,Manifest schema and domain rules,1712,1706
8,deduplication,country_records.json,remove_exact_duplicates,Exact duplicate records removed,1706,1704
9,reading,country_records.parquet,read_and_map,SHA-256 verified; canonical columns mapped,1712,1712


,check,expected,actual,tolerance,pass
0,country_records.csv: raw rows,1712,1712,0,True
1,country_records.csv: rejected rows,6,6,0,True
2,country_records.csv: duplicates,2,2,0,True
3,country_records.csv: canonical rows,1704,1704,0,True
4,country_records.csv: reconciliation,True,True,0,True
5,country_records.xlsx: raw rows,1712,1712,0,True
6,country_records.xlsx: rejected rows,6,6,0,True
7,country_records.xlsx: duplicates,2,2,0,True
8,country_records.xlsx: canonical rows,1704,1704,0,True
9,country_records.xlsx: reconciliation,True,True,0,True


Validation checks passed: 40
Total validation checks: 40
All checks passed: True


## 4. Conclusion

The pipeline demonstrates a reproducible process for recovering, validating, comparing, and exporting historical data from four different file formats.

Input integrity checks, record-level validation, duplicate handling, cross-format comparisons, and export verification help ensure that the published canonical dataset is consistent.

The audit and validation reports provide evidence of the processing steps and outcomes.